In [1]:
import spacy
nlp = spacy.load("en_core_web_sm")

def parse_sentence(sentence):
    doc = nlp(sentence)
    for token in doc:
        print(f"{token.text:15} | dep: {token.dep_:12} | pos: {token.pos_:8} | head: {token.head.text}")
    return doc

doc = parse_sentence("The cat that was sitting on the mat looked very hungry yesterday.")



The             | dep: det          | pos: DET      | head: cat
cat             | dep: nsubj        | pos: NOUN     | head: looked
that            | dep: nsubj        | pos: PRON     | head: sitting
was             | dep: aux          | pos: AUX      | head: sitting
sitting         | dep: relcl        | pos: VERB     | head: cat
on              | dep: prep         | pos: ADP      | head: sitting
the             | dep: det          | pos: DET      | head: mat
mat             | dep: pobj         | pos: NOUN     | head: on
looked          | dep: ROOT         | pos: VERB     | head: looked
very            | dep: advmod       | pos: ADV      | head: hungry
hungry          | dep: acomp        | pos: ADJ      | head: looked
yesterday       | dep: npadvmod     | pos: NOUN     | head: looked
.               | dep: punct        | pos: PUNCT    | head: looked


In [2]:
# Tags we want to REMOVE during compression
REMOVE_DEPS = {"relcl", "prep", "advmod", "npadvmod", "advcl", "appos", "pobj", "amod"}

def compress_sentence(sentence):
    doc = nlp(sentence)
    
    # Find tokens to remove
    remove_indices = set()
    
    def mark_subtree(token):
        remove_indices.add(token.i)
        for child in token.children:
            mark_subtree(child)
    
    for token in doc:
        if token.dep_ in REMOVE_DEPS:
            mark_subtree(token)
    
    # Rebuild sentence with remaining tokens
    compressed = " ".join([token.text for token in doc if token.i not in remove_indices])
    
    # Fix spacing around punctuation
    compressed = compressed.replace(" .", ".").replace(" ,", ",").replace(" '", "'")
    
    return compressed

# Test it
original = "The cat that was sitting on the mat looked very hungry yesterday."
compressed = compress_sentence(original)

print(f"Original:   {original}")
print(f"Compressed: {compressed}")
print(f"Ratio:      {len(compressed.split()) / len(original.split()):.2f}")

Original:   The cat that was sitting on the mat looked very hungry yesterday.
Compressed: The cat looked hungry.
Ratio:      0.33


In [8]:
import requests

url = "https://github.com/google-research-datasets/sentence-compression/raw/master/data/sent-comp.train01.json.gz"
response = requests.get(url, allow_redirects=True)

print("Status code:", response.status_code)
print("Content length:", len(response.content))
print("First 100 bytes:", response.content[:100])

Status code: 200
Content length: 24661983
First 100 bytes: b'\x1f\x8b\x08\x08s\xcb\xecX\x00\x03sent-comp.train01.json\x00\xec\xfd\xdb\x92\xe3H\x92&\x08\xdf\xf7S@Rj$/\xca\x19\xee<\xf9\xa1WRJ"#\x0f\x95\xd5\x95\x99!\x119]\xd3\xfd\xcfJ4\x9c\x84;\x11N\x12\x0c\x02t\x0f\x8f\x95\x11\xd9\xbb\xdd\xdb\xb9\xdd\xcb\xb9\x9a\xe7\x98}'


In [9]:
import gzip
import json

content = gzip.decompress(response.content).decode('utf-8')
lines = content.strip().split('\n')

print("Total lines:", len(lines))
print("\nFirst line preview:")
first = json.loads(lines[0])
print("Keys:", first.keys())
print("\nFull first item:")
print(json.dumps(first, indent=2)[:1000])


Total lines: 17516260

First line preview:


JSONDecodeError: Expecting property name enclosed in double quotes: line 1 column 2 (char 1)

In [10]:
import gzip
import json

content = gzip.decompress(response.content).decode('utf-8')

# It's one big JSON object, not line by line
data_raw = json.loads(content)

print("Type:", type(data_raw))
if isinstance(data_raw, list):
    print("Total items:", len(data_raw))
    print("First item keys:", data_raw[0].keys())
elif isinstance(data_raw, dict):
    print("Keys:", data_raw.keys())

JSONDecodeError: Extra data: line 1393 column 1 (char 28099)

In [11]:
import gzip

content = gzip.decompress(response.content).decode('utf-8')

# Just print first 500 characters raw
print(content[:500])

{
  "graph": {
    "id": "0",
    "sentence": "Serge Ibaka -- the Oklahoma City Thunder forward who was born in the Congo but played in Spain -- has been granted Spanish citizenship and will play for the country in EuroBasket this summer, the event where spots in the 2012 Olympics will be decided.",
    "node": [ {
      "form": "ROOT",
      "word": [ {
        "id": -1,
        "form": "ROOT",
        "stem": "ROOT",
        "tag": "ROOT"
      } ],
      "gender": 0,
      "head_word_index": 


In [12]:
import gzip
import json

content = gzip.decompress(response.content).decode('utf-8')

# Split by double newline since each record is a separate JSON block
raw_items = content.strip().split('\n\n')

print(f"Total raw items: {len(raw_items)}")

data = []
for item in raw_items:
    try:
        obj = json.loads(item)
        source = obj['graph']['sentence']
        compressed = obj['compression']['text']
        if source and compressed:
            data.append({'source': source, 'compressed': compressed})
    except:
        continue

print(f"Successfully loaded: {len(data)} samples")
print("\n--- Sample 1 ---")
print("Source:     ", data[0]['source'])
print("Compressed: ", data[0]['compressed'])
print("\n--- Sample 2 ---")
print("Source:     ", data[1]['source'])
print("Compressed: ", data[1]['compressed'])

Total raw items: 20000
Successfully loaded: 20000 samples

--- Sample 1 ---
Source:      Serge Ibaka -- the Oklahoma City Thunder forward who was born in the Congo but played in Spain -- has been granted Spanish citizenship and will play for the country in EuroBasket this summer, the event where spots in the 2012 Olympics will be decided.
Compressed:  Serge Ibaka has been granted Spanish citizenship and will play in EuroBasket.

--- Sample 2 ---
Source:      MILAN -Catania held Roma to a 1-1 draw in Serie A on Wednesday as the teams played out the remaining 25 minutes of a game that was called off last month.
Compressed:  Catania held Roma to a 1 1 draw in Serie A.


In [13]:
print("Testing compression on real samples:\n")
for i in range(5):
    source = data[i]['source']
    reference = data[i]['compressed']
    our_output = compress_sentence(source)
    print(f"Source:    {source}")
    print(f"Reference: {reference}")
    print(f"Ours:      {our_output}")
    print("-" * 90)

Testing compression on real samples:

Source:    Serge Ibaka -- the Oklahoma City Thunder forward who was born in the Congo but played in Spain -- has been granted Spanish citizenship and will play for the country in EuroBasket this summer, the event where spots in the 2012 Olympics will be decided.
Reference: Serge Ibaka has been granted Spanish citizenship and will play in EuroBasket.
Ours:      Serge Ibaka -- -- has been granted citizenship and will play,.
------------------------------------------------------------------------------------------
Source:    MILAN -Catania held Roma to a 1-1 draw in Serie A on Wednesday as the teams played out the remaining 25 minutes of a game that was called off last month.
Reference: Catania held Roma to a 1 1 draw in Serie A.
Ours:      MILAN -Catania held Roma.
------------------------------------------------------------------------------------------
Source:    State Street Corporation, a provider of investment servicing, investment management an

In [14]:
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

r1_scores, r2_scores, rL_scores, ratios = [], [], [], []

for item in data[:200]:
    source = item['source']
    reference = item['compressed']
    prediction = compress_sentence(source)

    scores = scorer.score(reference, prediction)
    r1_scores.append(scores['rouge1'].fmeasure)
    r2_scores.append(scores['rouge2'].fmeasure)
    rL_scores.append(scores['rougeL'].fmeasure)

    orig_len = len(source.split())
    comp_len = len(prediction.split())
    if orig_len > 0:
        ratios.append(comp_len / orig_len)

print(f"Evaluated on 200 samples")
print(f"ROUGE-1:          {sum(r1_scores)/len(r1_scores):.4f}")
print(f"ROUGE-2:          {sum(r2_scores)/len(r2_scores):.4f}")
print(f"ROUGE-L:          {sum(rL_scores)/len(rL_scores):.4f}")
print(f"Avg compression:  {sum(ratios)/len(ratios):.2f}  (lower = more compressed)")

Evaluated on 200 samples
ROUGE-1:          0.6432
ROUGE-2:          0.5160
ROUGE-L:          0.6355
Avg compression:  0.38  (lower = more compressed)


In [15]:
import re

def compress_sentence_v2(sentence):
    doc = nlp(sentence)
    remove_indices = set()

    def mark_subtree(token):
        remove_indices.add(token.i)
        for child in token.children:
            mark_subtree(child)

    for token in doc:
        if token.dep_ in REMOVE_DEPS:
            mark_subtree(token)

    compressed = " ".join([token.text for token in doc if token.i not in remove_indices])

    # Better cleanup
    compressed = re.sub(r'\s+', ' ', compressed)
    compressed = re.sub(r'\s([.,;:!?])', r'\1', compressed)
    compressed = re.sub(r',+', ',', compressed)
    compressed = re.sub(r',\s*\.', '.', compressed)
    compressed = re.sub(r'\(\s*\)', '', compressed)
    compressed = re.sub(r'--\s*--', '', compressed)
    compressed = compressed.strip(" ,")
    if not compressed.endswith('.'):
        compressed += '.'

    return compressed

# Test v2
print("Improved compression:\n")
for i in range(5):
    source = data[i]['source']
    reference = data[i]['compressed']
    our_output = compress_sentence_v2(source)
    print(f"Source:    {source}")
    print(f"Reference: {reference}")
    print(f"Ours v2:   {our_output}")
    print("-" * 90)

Improved compression:

Source:    Serge Ibaka -- the Oklahoma City Thunder forward who was born in the Congo but played in Spain -- has been granted Spanish citizenship and will play for the country in EuroBasket this summer, the event where spots in the 2012 Olympics will be decided.
Reference: Serge Ibaka has been granted Spanish citizenship and will play in EuroBasket.
Ours v2:   Serge Ibaka  has been granted citizenship and will play.
------------------------------------------------------------------------------------------
Source:    MILAN -Catania held Roma to a 1-1 draw in Serie A on Wednesday as the teams played out the remaining 25 minutes of a game that was called off last month.
Reference: Catania held Roma to a 1 1 draw in Serie A.
Ours v2:   MILAN -Catania held Roma.
------------------------------------------------------------------------------------------
Source:    State Street Corporation, a provider of investment servicing, investment management and investment research

In [16]:
code = '''
import spacy
import re

nlp = spacy.load("en_core_web_sm")

REMOVE_DEPS = {"relcl", "prep", "advmod", "npadvmod", "advcl", "appos", "pobj", "amod"}

def compress_sentence(sentence):
    doc = nlp(sentence)
    remove_indices = set()

    def mark_subtree(token):
        remove_indices.add(token.i)
        for child in token.children:
            mark_subtree(child)

    for token in doc:
        if token.dep_ in REMOVE_DEPS:
            mark_subtree(token)

    compressed = " ".join([token.text for token in doc if token.i not in remove_indices])

    compressed = re.sub(r\'\\s+\', \' \', compressed)
    compressed = re.sub(r\'\\s([.,;:!?])\', r\'\\1\', compressed)
    compressed = re.sub(r\',+\', \',\', compressed)
    compressed = re.sub(r\',\\s*\\.\', \'.\', compressed)
    compressed = re.sub(r\'\\(\\s*\\)\', \'\', compressed)
    compressed = re.sub(r\'--\\s*--\', \'\', compressed)
    compressed = compressed.strip(" ,")
    if not compressed.endswith(\'.\'):
        compressed += \'.\'

    return compressed
'''

with open("compressor.py", "w") as f:
    f.write(code)

print("Saved compressor.py successfully!")


Saved compressor.py successfully!
